# NCS offshore HAZOP, ESD and HIPPS engineering study

This notebook extends the complete offshore process from the recent engineering-study PR. The same process and governed engineering project drive the detailed P&ID proposal, DEXPI exchange, HAZOP preparation, ESD/HIPPS transient evidence and preliminary SIL verification. Results are workshop and SRS inputs—not approved detailed engineering or construction documents.

In [ ]:
%run ./complete_offshore_process_engineering_study.ipynb
print('Reusing real offshore process:', process.getName(), '| units:', process.getUnitOperations().size())

## HAZOP-ready detailed P&ID package

The complete proposal profile adds control, isolation, shutdown, blowdown, non-return, drain, vent and relief functions. HAZOP nodes are generated from this exact P&ID-backed model and fail readiness if an equipment node or element reference is unresolved.

In [ ]:
PidBasis = JClass('neqsim.process.engineering.pid.PidDesignBasis')
PidCatalog = JClass('neqsim.process.engineering.pid.NorsokPidRuleCatalog')
PidSynthesizer = JClass('neqsim.process.engineering.pid.PidDesignSynthesizer')
PidExporter = JClass('neqsim.process.engineering.pid.PidEngineeringPackageExporter')
ProtectedItem = JClass('neqsim.process.safety.overpressure.ProtectedItem')
ReliefScenario = JClass('neqsim.process.safety.overpressure.ReliefScenario')
ReliefCause = JClass('neqsim.process.safety.overpressure.ReliefCause')
ReliefPhase = JClass('neqsim.process.safety.overpressure.ReliefPhase')
OverpressureStudy = JClass('neqsim.process.safety.overpressure.OverpressureProtectionStudy')
ReliefInput = JClass('neqsim.process.engineering.ReliefDeviceDesignInput')
protected = ProtectedItem('20-VA-01', 45.0).setReliefSetPressureBara(43.0)
scenario = ReliefScenario.Builder('HP separator blocked gas outlet', ReliefCause.BLOCKED_OUTLET).phase(ReliefPhase.VAPOUR).reliefRateKgPerS(18.0).reliefTemperatureK(343.15).molarMassKgPerMol(0.022).compressibility(0.93).specificHeatRatio(1.24).build()
project.addOverpressureStudy(OverpressureStudy(protected).addScenario(scenario))
project.addReliefDeviceDesignInput(ReliefInput('20-PSV-101', '20-VA-01').setSelectedOrificeAreaIn2(11.05).setInletPiping(0.10, 3.0, 2.0).setOutletPiping(0.20, 35.0, 5.0).setConcurrencyGroup('FIRE-ZONE-20').setEvidenceReference('PRELIMINARY-PSV-DESIGN-CASE'))
pid = PidSynthesizer.synthesize(project, PidBasis('NORSOK-COMPLETE-PID-PROPOSALS', '20'), PidCatalog.completeProposals())
advanced_dir = package_root / 'revision-b' / 'ncs-hazop-safety-study'
advanced_dir.mkdir(parents=True, exist_ok=True)
pid_export = PidExporter.export(project, pid, Paths.get(str(advanced_dir)))
hazop = json.loads((advanced_dir / 'pid-hazop-study.json').read_text(encoding='utf-8'))
pid_json = json.loads((advanced_dir / 'pid-design-model.json').read_text(encoding='utf-8'))
print('P&ID elements:', len(pid_json['elements']), '| HAZOP nodes:', len(hazop['nodes']), '| ready:', hazop['readyForHazopWorkshop'])
assert hazop['readyForHazopWorkshop'] and len(pid_json['elements']) > 100

## Dynamic ESD and HIPPS final-element tests with SIL evaluation

The isolation tests use NeqSim transient execution and the same representative HP wellstream fluid. The HIPPS SIF separately records 2oo3 voting, SIL 3 PFDavg and proof-test assumptions. A successful simulation is evidence for response-time and pressure criteria only; it does not prove IEC 61511 compliance.

In [ ]:
ESDValve = JClass('neqsim.process.equipment.valve.ESDValve')
ESDLogic = JClass('neqsim.process.logic.esd.ESDLogic')
TripValveAction = JClass('neqsim.process.logic.action.TripValveAction')
TagMap = JClass('neqsim.process.operations.OperationalTagMap')
TagBinding = JClass('neqsim.process.operations.OperationalTagBinding')
TagRole = JClass('neqsim.process.measurementdevice.InstrumentTagRole')
TestPlan = JClass('neqsim.process.safety.esd.EmergencyShutdownTestPlan')
Criterion = JClass('neqsim.process.safety.esd.EmergencyShutdownTestCriterion')
TestRunner = JClass('neqsim.process.safety.esd.EmergencyShutdownTestRunner')
SafetyFunction = JClass('neqsim.process.safety.risk.sis.SafetyInstrumentedFunction')
SifCategory = JClass('neqsim.process.safety.risk.sis.SafetyInstrumentedFunction$SIFCategory')
TransientVerification = JClass('neqsim.process.engineering.safety.SafetyFunctionTransientVerification')
NcsStudy = JClass('neqsim.process.engineering.safety.NcsSafetyFunctionStudy')

def shutdown_test(name, stroke_time):
    feed = Stream(name + '-feed', process.getUnit('well stream').getThermoSystem().clone())
    feed.setFlowRate(BASE['feed_rate_kmol_h'] * 1000.0 / 3600.0, 'mol/sec'); feed.setPressure(BASE['Psep1_barg'] + 0.5, 'barg')
    valve = ESDValve(name + '-XV', feed); valve.setStrokeTime(stroke_time); valve.setCv(1500.0); valve.energize(); valve.setPercentValveOpening(100.0)
    vessel = Separator(name + '-HP-separator', valve.getOutletStream())
    dynamic_process = ProcessSystem(name); dynamic_process.add(feed); dynamic_process.add(valve); dynamic_process.add(vessel); dynamic_process.run()
    logic = ESDLogic(name); logic.addAction(TripValveAction(valve), 0.0)
    tags = TagMap().addBinding(TagBinding.builder('xv_opening').pidReference(name + '/XV').automationAddress(name + '-XV.percentValveOpening').unit('%').role(TagRole.BENCHMARK).build()).addBinding(TagBinding.builder('protected_pressure').pidReference(name + '/PT').automationAddress(name + '-HP-separator.gasOutStream.pressure').unit('bara').role(TagRole.BENCHMARK).build())
    plan = TestPlan.builder(name + ' dynamic isolation').duration(8.0).timeStep(0.5).tagMap(tags).enableLogic(name).triggerLogic(name).criterion(Criterion.finalAtMost(name + '-CLOSED', 'xv_opening', 1.0, '%').withClause('NORSOK S-001')).criterion(Criterion.maxAtMost(name + '-MAWP', 'protected_pressure', 45.0, 'bara').withClause('API 521 / NORSOK S-001')).criterion(Criterion.noSimulationErrors(name + '-NO-ERRORS')).standardReference('IEC 61511').evidenceReference('P&ID ' + name).build()
    return TestRunner.run(dynamic_process, plan, logic)

esd_result = shutdown_test('ESD-20-001', 4.0)
hipps_result = shutdown_test('HIPPS-20-001', 2.0)
(advanced_dir / 'esd-dynamic-test.json').write_text(str(esd_result.toJson()), encoding='utf-8')
(advanced_dir / 'hipps-dynamic-test.json').write_text(str(hipps_result.toJson()), encoding='utf-8')
samples = list(hipps_result.getTimeSeries())
times = [float(s.getTimeSeconds()) for s in samples]; pressure = [float(s.getValues().get('protected_pressure')) for s in samples]; opening = [float(s.getValues().get('xv_opening')) for s in samples]
dynamic_evidence = TransientVerification('SIF-HIPPS-20-001', 32.0, 45.0, 2.5).verify(jdouble(times), jdouble(pressure), jdouble(opening))
hipps_sif = SafetyFunction.builder().id('SIF-HIPPS-20-001').name('HP inlet HIPPS').description('Prevent HP separator overpressure').sil(3).pfd(5.0e-4).testIntervalHours(8760.0).mttr(24.0).addProtectedEquipment('20-VA-01').initiatingEvent('Blocked outlet or upstream control failure').safeState('Two series HIPPS isolation valves closed').category(SifCategory.HIPPS).architecture('2oo3').build()
esd_sif = SafetyFunction.builder().id('SIF-ESD-20-001').name('HP inlet ESD').description('Isolate HP production inlet').sil(2).pfd(5.0e-3).testIntervalHours(8760.0).mttr(24.0).addProtectedEquipment('20-VA-01').initiatingEvent('Confirmed process shutdown demand').safeState('Inlet ESD valve closed').category(SifCategory.ESD).architecture('1oo2').build()
study = NcsStudy(project.getProjectId()).add(hipps_sif, dynamic_evidence).add(esd_sif, dynamic_evidence)
(advanced_dir / 'ncs-safety-function-study.json').write_text(str(study.toJson()), encoding='utf-8')
print('ESD verdict:', esd_result.getVerdict(), '| HIPPS final-element verdict:', hipps_result.getVerdict(), '| transient criteria:', dict(dynamic_evidence))
assert str(esd_result.getVerdict()) == 'PASS' and str(hipps_result.getVerdict()) == 'PASS' and bool(dynamic_evidence.get('passed'))

## Professional PyDEXPI review sheets

The images below are rendered by PyDEXPI from the exported exchange model. Valve symbols are DEXPI piping components; instruments and safety functions retain proposal identity and approval state.

In [ ]:
from IPython.display import SVG
advanced_model = ProteusSerializer().load(str(advanced_dir), 'plant-pydexpi.xml')
DrawDiagram(advanced_model.diagram, padding=5.0, pretty=True).save_svg('ncs-detailed-pid', str(advanced_dir))
advanced_svg = advanced_dir / 'ncs-detailed-pid.svg'
display(SVG(filename=str(advanced_svg)))
outputs = sorted(p.name for p in advanced_dir.iterdir() if p.is_file())
display(pd.DataFrame({'generated artifact': outputs}))
print('PyDEXPI groups:', len(getattr(advanced_model.diagram, 'groups', [])), '| SVG bytes:', advanced_svg.stat().st_size)
assert advanced_svg.stat().st_size > 0 and 'pid-hazop-study.json' in outputs and 'ncs-safety-function-study.json' in outputs